In [2]:
# Cell 2: Start Ollama Background Server on an EMPTY GPU
import subprocess
import time
import os

print("🚀 Starting Ollama background server on GPU 5...")
ollama_path = os.path.expanduser("~/.local/bin/ollama")

# 🟢 Force Ollama to only see GPU 5 so it stays out of Jupyter's way
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "7" 

try:
    subprocess.Popen([ollama_path, "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3) 
    print("✅ Ollama server is now running on GPU 5!")
except Exception as e:
    print(f"❌ Failed to start Ollama: {e}")

🚀 Starting Ollama background server on GPU 5...
✅ Ollama server is now running on GPU 5!


In [3]:
# Cell 1: Setup, Models, and DB Connection
import os
import time
import logging

# 🛑 1. Set environment variables FIRST to fix PyTorch/CUDA states
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6" # 👈 Locked to ONE GPU for maximum speed

# 🟢 2. NOW it is safe to import heavy libraries
import re
import json
import torch
import chromadb
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

# --- SETUP LOGGING ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    handlers=[
        logging.FileHandler("pipeline_execution.log", encoding='utf-8'),
        logging.StreamHandler() # Also prints to console
    ]
)
logger = logging.getLogger(__name__)

logger.info(f"Torch Version: {torch.__version__}")
if torch.cuda.is_available():
    logger.info(f"✅ GPU Locked To: {torch.cuda.get_device_name(0)}")
else:
    logger.error("❌ No GPU detected.")

# 3. Connect to existing ChromaDB
logger.info("🔗 Connecting to ChromaDB...")
client = chromadb.PersistentClient(path="poetry_db")
collection = client.get_collection(name="hindwi_poems") 

# 4. Load Embedder
logger.info("⏳ Loading Embedder (multilingual-e5-large)...")
embedder = SentenceTransformer('intfloat/multilingual-e5-large', device='cuda')

# 5. Load Param2 MoE LLM
logger.info("⏳ Loading Param2-17B-A2.4B-Thinking...")
MODEL_ID = "bharatgenai/Param2-17B-A2.4B-Thinking"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    device_map="cuda", # 👈 Back to 'cuda' since we only have 1 visible GPU now
    low_cpu_mem_usage=True,
    trust_remote_code=True  
)
model.eval()
logger.info("✅ All models loaded and ready for experiments!")

/home/literature/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-24 10:27:30,423 | INFO     | Torch Version: 2.9.1+cu128
2026-02-24 10:27:30,679 | INFO     | ✅ GPU Locked To: NVIDIA RTX A6000
2026-02-24 10:27:30,680 | INFO     | 🔗 Connecting to ChromaDB...
2026-02-24 10:27:30,690 | INFO     | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-02-24 10:27:30,801 | INFO     | ⏳ Loading Embedder (multilingual-e5-large)...
2026-02-24 10:27:30,803 | INFO     | Load pretrained SentenceTransformer: intfloat/multilingual-e5-large
2026-02-24 10:27:37,317 | INFO     | ⏳ Loading Param2-17B-A2.4B-Thinking...
You are using a model of type param2moe to instantiate a model of type . This is not supported for all configurations of models a

In [4]:
# Cell 3: PoetryExperimenter Class
class PoetryExperimenter:
    def __init__(self, model, tokenizer, collection, embedder, logger):
        self.model = model
        self.tokenizer = tokenizer
        self.collection = collection
        self.embedder = embedder
        self.logger = logger # Pass the logger in

    def _generate(self, system_prompt, user_prompt, temperature=0.6): # Lowered default temp
        """Helper for standard generation with reasoning parsed out and context limits respected."""
        start_time = time.time()
        
        # Make the system prompt slightly stricter to prevent overthinking
        strict_system = system_prompt + " अपनी सोच प्रक्रिया को <think> और </think> टैग के अंदर रखें। टैग के बाहर केवल कविता लिखें, कोई अन्य स्पष्टीकरण नहीं।"
        
        messages = [
            {"role": "system", "content": strict_system},
            {"role": "user", "content": user_prompt}
        ]
        
        inputs = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(self.model.device)
        
        input_length = inputs.shape[1]
        MAX_MODEL_CONTEXT = 4096
        safe_max_new_tokens = MAX_MODEL_CONTEXT - input_length - 10 
        
        if safe_max_new_tokens < 500:
            self.logger.warning(f"⚠️ Prompt is very long ({input_length} tokens). Output might get cut off.")
            safe_max_new_tokens = 500 
            
        self.logger.info(f"   [Inference] Prompt: {input_length} tokens | Allowed Gen: {safe_max_new_tokens} tokens")
            
        with torch.no_grad():
            outputs = self.model.generate(
                inputs, 
                max_new_tokens=safe_max_new_tokens, 
                temperature=0.6,         # 0.6 पर थिंकिंग मॉडल सबसे स्टेबल होते हैं
                top_p=0.9,
                top_k=40,                # 🟢 NEW FIX: यह मॉडल को डिक्शनरी छापने से रोकेगा
                do_sample=True, 
                repetition_penalty=1.0,  # 🟢 FIX: इसे 1.0 (बंद) ही रखें ताकि मॉडल पागल न हो
                eos_token_id=self.tokenizer.eos_token_id,
                pad_token_id=self.tokenizer.pad_token_id
            )
            
        raw_output = self.tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
        
        think_match = re.search(r'<think>(.*?)</think>', raw_output, flags=re.DOTALL)
        if think_match:
            thoughts = think_match.group(1).strip()
            self.logger.info(f"   [Thoughts] Model reasoned for {len(thoughts.split())} words.")
            self.logger.debug(f"   [Hidden Reasoning]:\n{thoughts}")
            
        # 2. Strip out the <think> blocks
        clean_output = re.sub(r'<think>.*?</think>', '', raw_output, flags=re.DOTALL).strip()
        
        lines = clean_output.split('\n')
        last_header_idx = -1
        
        for i, line in enumerate(lines):
            if line.strip().startswith('#'):
                last_header_idx = i
                
        if last_header_idx != -1:
            clean_output = '\n'.join(lines[last_header_idx+1:]).strip()
            
        final_lines = [line for line in clean_output.split('\n') if not (line.strip().startswith('* ') or line.strip().startswith('- '))]
        clean_output = '\n'.join(final_lines).strip()
        
        elapsed_time = time.time() - start_time
        self.logger.info(f"   [Inference] Completed in {elapsed_time:.2f} seconds.")
        
        return clean_output

    def find_best_poet(self, topic):
        try:
            query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
            results = self.collection.query(query_embeddings=query_vector, n_results=1)
            if results['metadatas'] and results['metadatas'][0]:
                poet = results['metadatas'][0][0]['poet_slug']
                self.logger.info(f"   [RAG] Found best poet match: {poet}")
                return poet
        except Exception as e:
            self.logger.error(f"⚠️ Error finding best poet: {e}")
        return "ramdhari-singh-dinkar" 

    
    def zero_shot(self, topic):
        return self._generate("आप एक उत्कृष्ट हिंदी कवि हैं।", f"विषय: '{topic}' पर एक कविता लिखें।")

    def few_shot(self, topic, style="ramdhari-singh-dinkar"):
        query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
        res = self.collection.query(query_embeddings=query_vector, n_results=2, where={"poet_slug": style})
        
        examples_text = ""
        if res['documents'] and res['documents'][0]:
            for i, text in enumerate(res['documents'][0]):
                example_lines = "\n".join(text.split("\n")[:6]).strip()
                examples_text += f"उदाहरण {i+1}:\n{example_lines}\n\n"
                
        # सौम्य प्रॉम्प्ट जो बिना सख्ती के उसे ओरिजिनल होने को कहता है
        sys = "आप एक प्रख्यात हिंदी कवि हैं। आपका काम दी गई शैली को समझना और उसी अंदाज़ में एक नई रचना करना है।"
        user = (
            f"यहाँ कुछ उदाहरण दिए गए हैं:\n{examples_text}\n"
            f"अब, '{topic}' विषय पर 4 से 8 पंक्तियों की एक नई और मौलिक (original) कविता लिखें। "
            f"ध्यान रहे, आपको उदाहरणों की पंक्तियाँ नहीं दोहरानी हैं, केवल उनकी भावना और लय का उपयोग करके नए शब्द लिखने हैं।"
        )
        return self._generate(sys, user)

    def rag_style_conditioned(self, topic, style="ramdhari-singh-dinkar"):
        query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
        res = self.collection.query(query_embeddings=query_vector, n_results=2, where={"poet_slug": style})
        
        context = ""
        if res['documents'] and res['documents'][0]:
            for i, text in enumerate(res['documents'][0]):
                context += f"संदर्भ {i+1}:\n{text[:300]}...\n"
                
        user = (
            f"इस शैली का गहराई से अध्ययन करें:\n{context}\n\n"
            f"अब '{topic}' विषय पर एक बिल्कुल नई कविता लिखें। "
            f"संदर्भ से पंक्तियाँ न चुराएं, बल्कि अपनी कल्पना से उसी शैली में नए शब्द पिरोएं।"
        )
        return self._generate("आप एक रचनात्मक और मौलिक (original) कवि हैं।", user)
    
    # 🔹 4. Chain-of-Thought & 5. Plan-Then-Generate
    def plan_then_generate(self, topic):
        sys = "आप एक कवि और आलोचक हैं। पहले कविता की योजना बनाएं, फिर कविता लिखें।"
        user = (
            f"विषय: '{topic}'\n"
            "1. भाव (Mood) तय करें।\n"
            "2. 5 मुख्य शब्द (Vocabulary) चुनें।\n"
            "3. अलंकार (Metaphor) सोचें।\n"
            "4. अंत में 'कविता:' शीर्षक के साथ कविता लिखें।"
        )
        return self._generate(sys, user)

    # 🔹 6. Self-Critique Loop
    def self_critique(self, topic):
        draft = self.zero_shot(topic)
        critique_prompt = f"इस कविता की आलोचना करें और 2 कमियां निकालें (लय या शब्द चयन):\n{draft}"
        critique = self._generate("आप एक कठोर आलोचक हैं।", critique_prompt)
        
        refine_prompt = f"मूल कविता:\n{draft}\n\nआलोचना:\n{critique}\n\nआलोचना को ध्यान में रखते हुए एक श्रेष्ठ संस्करण लिखें।"
        final = self._generate("आप एक मास्टर कवि हैं जो अपनी गलतियों को सुधारता है।", refine_prompt)
        return f"--- DRAFT ---\n{draft}\n\n--- CRITIQUE ---\n{critique}\n\n--- FINAL ---\n{final}"

    # 🔹 7. Constraint-Based
    def constraint_based(self, topic):
        user = (
            f"विषय: '{topic}'\n"
            "नियम:\n"
            "1. कविता में ठीक 4 पंक्तियां (lines) होनी चाहिए।\n"
            "2. 'आसमान' शब्द का प्रयोग वर्जित है।\n"
            "3. अंतिम पंक्ति 'कहानी' शब्द पर खत्म होनी चाहिए।"
        )
        return self._generate("आप नियमों का सख्ती से पालन करने वाले कवि हैं।", user)

    # 🔹 8. Temperature Experiments
    def temp_experiment(self, topic):
        low_temp = self._generate("आप एक कवि हैं।", f"विषय: '{topic}'", temperature=0.2)
        high_temp = self._generate("आप एक कवि हैं।", f"विषय: '{topic}'", temperature=1.2)
        return f"--- TEMP 0.2 (Predictable) ---\n{low_temp}\n\n--- TEMP 1.2 (Creative/Chaotic) ---\n{high_temp}"

    # 🔹 9. Persona-Based
    def persona_based(self, topic):
        sys = "आप 19वीं सदी के एक उदास, दार्शनिक कवि हैं जो पुरानी हिंदी (तद्भव/तत्सम बहुल) में लिखते हैं।"
        return self._generate(sys, f"इस विषय पर अपने विचार प्रकट करें: '{topic}'")

    # 🔹 11. Prompt Engineering Variants
    def prompt_variants(self, topic):
        variant_a = self._generate("कवि बनो।", f"{topic} पर लिखो।")
        variant_b = self._generate(
            "आप साहित्य अकादमी पुरस्कार विजेता हैं। आपकी भाषा हृदय को छू लेने वाली और प्रतीकात्मक है।", 
            f"कृपया '{topic}' विषय पर एक मर्मस्पर्शी रचना प्रस्तुत करें।"
        )
        return f"--- BASIC PROMPT ---\n{variant_a}\n\n--- ENGINEERED PROMPT ---\n{variant_b}"

    # 🔹 12. Multi-Agent Poetry (Model talks to itself)
    def multi_agent(self, topic):
        sys_a = "आप 'कवि A' हैं। आप बहुत ही शांत और प्रकृति-प्रेमी हैं। विषय पर केवल पहली 4 पंक्तियां (Stanza 1) लिखें।"
        stanza_1 = self._generate(sys_a, f"विषय: '{topic}'")
        
        sys_b = "आप 'कवि B' हैं। आपका स्वभाव उग्र और क्रांतिकारी है। 'कवि A' की कविता को आगे बढ़ाते हुए अगली 4 पंक्तियां (Stanza 2) लिखें।"
        stanza_2 = self._generate(sys_b, f"कवि A ने यह लिखा है:\n{stanza_1}\n\nअब आप इसे अपने विद्रोही अंदाज में पूरा करें।")
        
        return f"--- STANZ 1 (Calm Agent) ---\n{stanza_1}\n\n--- STANZA 2 (Fiery Agent) ---\n{stanza_2}"
    def auto_eval(self, topic, generated_poem, poet_name, reference_poems, num_evals=5):
        # ... (Keep your auto_eval function exactly the same) ...
        # Just add a quick log at the start:
        self.logger.info("   [Evaluation] Starting Dual-LLM evaluation via Ollama...")
        import json
        import ollama
        import re
        import statistics

        system_instruction = (
            "You are an expert Hindi literary critic and NLP evaluation judge. "
            "Your task is to evaluate a generated Hindi poem. "
            "You must output ONLY a valid JSON object. Do not include markdown formatting, explanations, or introductory text."
        )
        
        evaluation_prompt = (
            f"Topic: {topic}\n"
            f"Target Poet Style: {poet_name}\n\n"
            f"--- REFERENCE POEMS BY {poet_name} ---\n"
            f"{reference_poems}\n\n"
            f"--- GENERATED POEM TO EVALUATE ---\n"
            f"{generated_poem}\n\n"
            "Evaluate the generated poem on a scale of 1 to 10 for the following metrics:\n"
            "1. 'fluency': Language fluency, correct Hindi grammar, and natural rhythm.\n"
            "2. 'coherence': Logical flow, structural integrity, and how well the stanzas connect.\n"
            "3. 'relevance': How accurately it addresses the exact Topic.\n"
            "4. 'creativity': Originality of metaphors, vivid imagery, and avoiding cliches.\n"
            "5. 'style_similarity': How closely the vocabulary, tone, and sentence structure match the Reference Poems provided above.\n\n"
            "Return EXACTLY this JSON format and nothing else:\n"
            '{"fluency": 0, "coherence": 0, "relevance": 0, "creativity": 0, "style_similarity": 0}'
        )

        final_scores = {"Llama-3.1": {}, "Gemma-2": {}}
        metrics = ['fluency', 'coherence', 'relevance', 'creativity', 'style_similarity']

        def clean_json(text):
            text = text.strip()
            if text.startswith("```json"): text = text[7:]
            if text.endswith("```"): text = text[:-3]
            match = re.search(r'\{.*\}', text, re.DOTALL)
            return match.group(0) if match else text

        for model_name, dict_key in [('llama3.1', 'Llama-3.1'), ('gemma2', 'Gemma-2')]:
            raw_scores = {m: [] for m in metrics}
            
            for i in range(num_evals):
                try:
                    response = ollama.chat(model=model_name, messages=[
                        {'role': 'system', 'content': system_instruction},
                        {'role': 'user', 'content': evaluation_prompt}
                    ], options={'temperature': 0.7})
                    
                    clean_text = clean_json(response['message']['content'])
                    parsed = json.loads(clean_text)
                    
                    for m in metrics:
                        if m in parsed:
                            raw_scores[m].append(float(parsed[m]))
                except Exception as e:
                    pass 
            
            aggregated = {}
            for m in metrics:
                vals = raw_scores[m]
                if len(vals) > 1:
                    aggregated[m] = {
                        "mean": round(statistics.mean(vals), 2),
                        "std_dev": round(statistics.stdev(vals), 2)
                    }
                elif len(vals) == 1:
                    aggregated[m] = {"mean": round(vals[0], 2), "std_dev": 0.0}
                else:
                    aggregated[m] = {"error": "Failed all 5 attempts"}
            
            final_scores[dict_key] = aggregated

        return json.dumps(final_scores, indent=2, ensure_ascii=False)


    def run_all(self, topics):
        experiments = {
            "Zero-Shot": self.zero_shot,
            "Few-Shot": self.few_shot,
            "RAG Style (Best Match)": self.rag_style_conditioned,
            "Plan-Then-Generate": self.plan_then_generate,
            "Self-Critique": self.self_critique,
            "Constraints": self.constraint_based,
            "Temperature (0.2 vs 1.2)": self.temp_experiment,
            "Persona (19th Century)": self.persona_based,
            "Prompt Variants": self.prompt_variants,
            "Multi-Agent": self.multi_agent
        }

        output_dir = "Param_Output"
        os.makedirs(output_dir, exist_ok=True)
        self.logger.info(f"📂 Execution started. Saving to '{output_dir}/'")
        
        for topic in topics:
            self.logger.info(f"\n{'='*50}\n🌟 STARTING TOPIC: {topic}\n{'='*50}")
            
            best_poet_slug = self.find_best_poet(topic)
            safe_filename = re.sub(r'[^\w\s-]', '', topic).strip().replace(' ', '_')
            file_path = os.path.join(output_dir, f"{safe_filename}.txt")
            
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(f"🌟 TOPIC: {topic}\n")
                f.write(f"🎯 BEST POET MATCH: '{best_poet_slug}'\n")
                f.write(f"{'='*50}\n\n")
                
                for exp_name, exp_func in experiments.items():
                    self.logger.info(f"🧪 Running Experiment: {exp_name}")
                    f.write(f"🧪 EXPERIMENT: {exp_name}\n")
                    f.write(f"{'-'*50}\n")
                    
                    try:
                        # 1. Fetch Reference Context
                        query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
                        res = self.collection.query(query_embeddings=query_vector, n_results=3, where={"poet_slug": best_poet_slug})
                        
                        reference_context = ""
                        if res['documents'] and res['documents'][0]:
                            for i, text in enumerate(res['documents'][0]):
                                reference_context += f"Reference {i+1}:\n{text[:300]}...\n"

                        # 2. Generate
                        if exp_name in ["RAG Style (Best Match)", "Few-Shot"]:
                            poem = exp_func(topic, style=best_poet_slug)
                        else:
                            poem = exp_func(topic)
                        
                        # 3. Evaluate
                        eval_scores_json = self.auto_eval(
                            topic=topic, 
                            generated_poem=poem, 
                            poet_name=best_poet_slug, 
                            reference_poems=reference_context
                        )
                        
                        # 4. Save
                        f.write("📜 GENERATED POEM:\n")
                        f.write(poem + "\n\n")
                        f.write("🧠 DUAL-AI EVALUATION (JSON):\n")
                        f.write(eval_scores_json + "\n\n")
                        f.write(f"{'='*50}\n\n")
                        
                    except Exception as e:
                        self.logger.error(f"❌ FAILED! Error in {exp_name}: {e}")
                        f.write(f"❌ ERROR GENERATING POEM: {str(e)}\n\n")
                        f.write(f"{'='*50}\n\n")
            
            self.logger.info(f"💾 Completed topic. Saved to: {file_path}")

        self.logger.info("✅ All experiments complete!")

In [5]:
# # Quick Debug Run
# TEST_TOPICS = ["🌿 प्रकृति (Nature): बारिश की पहली बूंद"]

# # 🟢 FIX: Added 'logger' to the end of this list
# experimenter = PoetryExperimenter(model, tokenizer, collection, embedder, logger)
# experimenter.run_all(TEST_TOPICS)
# Cell 3: Run the Massive Experiment Suite
TEST_TOPICS = [
    "🌿 प्रकृति (Nature): बारिश की पहली बूंद",
    "❤️ भावनात्मक (Emotional): अधूरी मोहब्बत",
    "🌍 सामाजिक (Social): नारी शक्ति",
    "🧠 दार्शनिक (Philosophical): समय का चक्र",
    "🎭 रचनात्मक / अनोखा (Creative): टूटी हुई घड़ी की कहानी",
    "🌅 आशावादी (Optimistic): ख्वाबों का आसमान",
    "😔 उदास (Sad): सूनी राहें",
    "🌾 नॉस्टैल्जिक (Nostalgic): मिट्टी की खुशबू",
    "💔 दर्दभरा (Painful): चुप्पी का बोझ",
    "✨ प्रेरणादायक (Inspirational): उम्मीद की किरण",
    "🏡 स्मृतिपूर्ण (Reminiscent): बचपन की गलियाँ",
    "🌆 अकेलापन (Lonely): अजनबी शहर",
    "❤️ रोमांटिक (Romantic): दिल की दस्तक",
    "🤝 भावुक (Emotional): रिश्तों की डोर",
    "⏳ दार्शनिक (Reflective): वक्त की रेत",
    "🕊️ उत्साहपूर्ण (Energetic): सपनों की उड़ान",
    "🎈 निराशाजनक (Hopeless): टूटी हुई पतंग",
    "🪞 गंभीर (Serious): सच का आईना",
    "📜 विरहपूर्ण (Separation): आख़िरी ख़त",
    "🌄 सकारात्मक (Positive): नई सुबह"
]

# Initialize and Run
experimenter = PoetryExperimenter(model, tokenizer, collection, embedder, logger)
experimenter.run_all(TEST_TOPICS)

2026-02-24 10:27:47,554 | INFO     | 📂 Execution started. Saving to 'Param_Output/'
2026-02-24 10:27:47,555 | INFO     | 
🌟 STARTING TOPIC: 🌿 प्रकृति (Nature): बारिश की पहली बूंद
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.91it/s]
2026-02-24 10:27:48,311 | INFO     |    [RAG] Found best poet match: gopalkrishna-kaul
2026-02-24 10:27:48,312 | INFO     | 🧪 Running Experiment: Zero-Shot
Batches: 100%|██████████| 1/1 [00:00<00:00, 54.16it/s]
2026-02-24 10:27:48,402 | INFO     |    [Inference] Prompt: 73 tokens | Allowed Gen: 4013 tokens
2026-02-24 10:28:10,441 | INFO     |    [Inference] Completed in 22.06 seconds.
2026-02-24 10:28:10,443 | INFO     |    [Evaluation] Starting Dual-LLM evaluation via Ollama...
2026-02-24 10:28:14,585 | INFO     | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-24 10:28:15,244 | INFO     | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-24 10:28:15,969 | INFO     | HTTP Request: POST http://127.0.0

In [6]:
# # Cell 3: Run the Massive Experiment Suite
# TEST_TOPICS = [
#     "🌿 प्रकृति (Nature): बारिश की पहली बूंद",
#     "❤️ भावनात्मक (Emotional): अधूरी मोहब्बत",
#     "🌍 सामाजिक (Social): नारी शक्ति",
#     "🧠 दार्शनिक (Philosophical): समय का चक्र",
#     "🎭 रचनात्मक / अनोखा (Creative): टूटी हुई घड़ी की कहानी",
#     "🌅 आशावादी (Optimistic): ख्वाबों का आसमान",
#     "😔 उदास (Sad): सूनी राहें",
#     "🌾 नॉस्टैल्जिक (Nostalgic): मिट्टी की खुशबू",
#     "💔 दर्दभरा (Painful): चुप्पी का बोझ",
#     "✨ प्रेरणादायक (Inspirational): उम्मीद की किरण",
#     "🏡 स्मृतिपूर्ण (Reminiscent): बचपन की गलियाँ",
#     "🌆 अकेलापन (Lonely): अजनबी शहर",
#     "❤️ रोमांटिक (Romantic): दिल की दस्तक",
#     "🤝 भावुक (Emotional): रिश्तों की डोर",
#     "⏳ दार्शनिक (Reflective): वक्त की रेत",
#     "🕊️ उत्साहपूर्ण (Energetic): सपनों की उड़ान",
#     "🎈 निराशाजनक (Hopeless): टूटी हुई पतंग",
#     "🪞 गंभीर (Serious): सच का आईना",
#     "📜 विरहपूर्ण (Separation): आख़िरी ख़त",
#     "🌄 सकारात्मक (Positive): नई सुबह"
# ]

# # Initialize and Run
# experimenter = PoetryExperimenter(model, tokenizer, collection, embedder)
# experimenter.run_all(TEST_TOPICS)